In [1]:
import os
import json
import glob

from dotenv import load_dotenv
from pathlib import Path
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field

In [2]:
from datasets import Dataset

In [3]:
from ragas import evaluate
#from ragas.metrics.collections import answer_relevancy, faithfulness, context_precision, context_recall
from ragas.metrics.collections import Faithfulness, AnswerAccuracy, ContextPrecision, ContextRecall

In [4]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [5]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.messages import SystemMessage, HumanMessage

### Vectorstore Creation

In [6]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [7]:
load_dotenv(override=True)

True

In [8]:
def load_json_with_root(filepath):
    with open(filepath, 'r') as f:
        full_data = json.load(f)
        policy_name = full_data.get("policy_name", "unknown")
        category = full_data.get("category", "unknown")
        source = full_data.get("source_path", "unknown").split('\\')[1]
        
    def metadata_func(record: dict, base_metadata: dict):
        base_metadata['policy_name'] = policy_name
        base_metadata['category'] = category
        base_metadata['source'] = source
        base_metadata['page_type'] = record.get("page_type", "unknown")
        return base_metadata
    
    return JSONLoader(
        file_path=filepath,
        jq_schema='.pages[] | select(.page_type == "content")',
        content_key='text',
        metadata_func=metadata_func
    )

In [9]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob='**/*.json', loader_cls=load_json_with_root)
    folder_docs = loader.load()
    
    for doc in folder_docs:
        documents.append(doc)
        
print(len(documents))

437


#### Different Text Splitters

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

1953

In [11]:
hf_embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

In [12]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=hf_embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=hf_embeddings, persist_directory=db_name)

### Test Case Generation

In [13]:
class TestQuestion(BaseModel):
    test_id: str = Field(description="Unique identifier for the test case")
    query: str = Field(description="The user question to be asked to the RAG system")
    source_doc: str = Field(description="The policy document(s) where the answer should come from")
    expected_answer: str = Field(description="The correct ground truth answer for evaluation")
    relevant_section: str = Field(description="The section in the document where the answer lives")
    question_type: str = Field(description="Category of the question")
    difficulty: str = Field(description="Complexity level of the question")

In [14]:
def load_tests() -> List[TestQuestion]:
    tests = []
    with open("tests.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
            test = json.loads(line.strip())
            tests.append(TestQuestion(**test))
            
    return tests

In [15]:
tests = load_tests()
len(tests)

60

### LLM Answers

In [16]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [17]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the LIC (Life Insurance Corporation of India).
You are chatting with a user about LIC's insurance products only.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [18]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [19]:
answer_question("What are the two death benefit options available under LIC Digi Term?", [])

'Under LIC Digi Term, the two death benefit options available are:\n\n1. **Lumpsum Payment:** The death benefit is paid in a lump sum to the nominee or legal heir.\n2. **InstaPay (or similar installment options):** The death benefit can also be paid in installments, as per the settlement option chosen by the policyholder at the time of policy inception.\n\nPlease note that the specific options may vary, and I recommend reviewing the policy document or consulting with an LIC advisor for detailed information tailored to your needs.'

### Ragas Evaluation

In [20]:
# faithfulness = Faithfulness()
# answer_relevancy = AnswerRelevancy()
# context_precision = ContextPrecision()
# context_recall = ContextRecall()

# metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

In [23]:
results_dict = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for tc in tests:
    answer = answer_question(tc.query, [])
    contexts = [doc.page_content for doc in retriever.invoke(tc.query)]
    #answer, contexts = rag_pipeline.run(tc.query)
    results_dict["question"].append(tc.query)
    results_dict["answer"].append(answer)
    results_dict["contexts"].append(contexts)
    results_dict["ground_truth"].append(tc.expected_answer)



    
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
ragas_llm = LangchainLLMWrapper(llm)
dataset = Dataset.from_dict(results_dict)

metrics=[Faithfulness(), AnswerAccuracy(), ContextPrecision(), ContextRecall()]

for metric in metrics:
    metric.llm = ragas_llm
    # Context metrics specifically require the embedding wrapper
    if hasattr(metric, 'embeddings'):
        metric.embeddings = ragas_embeddings

results = evaluate(dataset, llm=ragas_llm, embeddings=ragas_embeddings, 
                   metrics=metrics)

C:\Users\HP\AppData\Local\Temp\ipykernel_4392\2167921605.py:20: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)
C:\Users\HP\AppData\Local\Temp\ipykernel_4392\2167921605.py:21: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


TypeError: Faithfulness.__init__() missing 1 required positional argument: 'llm'

In [ ]:
results